## PULSAR NULLING FRACTION CALCULATOR

In [1]:
# Required packages: pip install numpy matplotlib scipy

# --- Configuration ---
pulsar_name = 'J1559-5545'     # Change this to your pulsar name
data_dir    = 'pulsar_results' # Path to the directory containing results folders

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import datetime
%matplotlib inline
from scipy.optimize import curve_fit

### READ IN THE OUTPUT FILE FROM NF_CALCULATIONS.PY:

In [ ]:
unzipped = np.loadtxt(f'{data_dir}/{pulsar_name}/{pulsar_name}.txt')

# unzipped[0] = the mjds
# unzipped[1] = the Bayes nulling fractions
# unzipped[2] = the lower uncertainty on the Bayes nulling fraction
# unzipped[3] = the upper uncertainty on the Bayes nulling fraction
# unzipped[4] = the S/N proxies
# unzipped[5] = the number of profiles in the observations
# unzipped[6] = the Histogram Scaling (HS) nulling fractions
# unzipped[7] = the maximum consecutive nulls in the observations

### READ IN THE OUTPUT FILE FOR THE LIST OF NULLS AND NON NULLS:

In [4]:
with open(f'{data_dir}/{pulsar_name}/{pulsar_name}_consecutive_nulls_and_non.txt', 'r') as f:
    all_nulls = eval(f.readline().strip())
    all_non_nulls = eval(f.readline().strip())

In [5]:
flat_all_nulls = [num for sublist in all_nulls for num in sublist]
flat_all_non_nulls = [num for sublist in all_non_nulls for num in sublist]
mean_train = np.mean(flat_all_nulls + flat_all_non_nulls)
#print(flat_all_nulls)
#print(flat_all_non_nulls)
#print(mean_train)

### CREATE A LIST OF THE NUMBER OF PROFILES IN THE OBSERVATIONS:

In [6]:
n_prof_list = unzipped[5]

### APPROXIMATION OF THE EXPECTED NULLING FRACTION OF THE PULSAR:

In [7]:
nf_exp = np.average(unzipped[1], weights=np.mean([unzipped[2],unzipped[3]],axis=0))
print(f'Approx. nulling fraction across all data: {round(nf_exp,3)}')

Approx. nulling fraction across all data: 0.251


### BAYESIAN NULLING FRACTION MEASUREMENT UNCERTAINTY:

In [8]:
bayes_uncertainties = np.array([unzipped[2],unzipped[3]])

In [9]:
###max_nulls = unzipped[7]

binom_unc_nf_list = []

###mean_of_max_nulls = np.mean(max_nulls)
###print(mean_of_max_nulls,mean_train)
for u in range(len(n_prof_list)):
    #n_trials = n_prof_list[u] / mean_of_max_nulls
    n_trials = n_prof_list[u] / mean_train
    binom_unc = np.sqrt(n_trials*nf_exp*(1-nf_exp))
    binom_unc_nf = binom_unc / n_trials
    binom_unc_nf_list.append(binom_unc_nf)
print(binom_unc_nf_list)

[0.036589721244004496, 0.036589721244004496, 0.036589721244004496, 0.036589721244004496, 0.036589721244004496, 0.036589721244004496, 0.036589721244004496, 0.036589721244004496, 0.036589721244004496, 0.036589721244004496, 0.036589721244004496, 0.036589721244004496, 0.036589721244004496, 0.036589721244004496, 0.036589721244004496, 0.036589721244004496, 0.036589721244004496, 0.036589721244004496, 0.036589721244004496, 0.036589721244004496, 0.036589721244004496, 0.036589721244004496, 0.036589721244004496, 0.036589721244004496, 0.036589721244004496, 0.036589721244004496, 0.036589721244004496, 0.08221204000233939, 0.08221204000233939]


### COMBINE BINOMAL UNCERTAINTY AND BAYESIAN NULLING FRACTION MEASUREMENT UNCERTAINTY:

In [10]:
lower_unc = []
upper_unc = []

for d in range(len(unzipped[2])):
    lower_unc.append(np.sqrt(binom_unc_nf_list[d]**2 + unzipped[2][d]**2))

for d in range (len(unzipped[3])):
    upper_unc.append(np.sqrt(binom_unc_nf_list[d]**2 + unzipped[3][d]**2))
    
lower_unc = np.array(lower_unc)
upper_unc = np.array(upper_unc)

uncertainties = np.array([lower_unc,upper_unc])

uncertainties_mean = np.sqrt((uncertainties[0]**2 + uncertainties[1]**2)/2)

print(uncertainties)

[[0.09011689 0.05685157 0.0558104  0.0433454  0.10511614 0.08732539
  0.12096078 0.07787407 0.05874696 0.05841736 0.1656845  0.1530847
  0.06172593 0.19829211 0.06169239 0.06134349 0.04446965 0.09150346
  0.14137943 0.06691662 0.04628423 0.06903423 0.06041364 0.05358593
  0.05164927 0.1203983  0.18677032 0.23332077 0.11473795]
 [0.06216429 0.05741202 0.06483433 0.04036481 0.08899703 0.14361775
  0.11995316 0.1060412  0.06042823 0.05757104 0.21649522 0.20646557
  0.1122393  0.24616375 0.05883155 0.05784572 0.08880889 0.19513968
  0.13251843 0.06489859 0.06468458 0.08734659 0.05900557 0.05457637
  0.05147535 0.13513907 0.15201256 0.26608938 0.24632982]]


### CREATE A BEST FIT LINE:

In [11]:
def linear_model(x, a, b):
    return a * x + b

### CONVERT ABSOLUTE MJD TO RELATIVE-TO-THE-FIRST-MJD:

In [12]:
mjd_delta = [mjd-unzipped[0][0] for mjd in unzipped[0]] # MJD - FIRST MJD

### CALCULATE THE BEST FIT LINE FOR THE BAYESIAN TECHNIQUE:

In [13]:
# Perform the curve fit
params, covariance = curve_fit(linear_model, mjd_delta, unzipped[1], sigma=uncertainties_mean, absolute_sigma=True)

# Extract the best-fit parameters
a_fit, b_fit = params

# Calculate the standard deviations (uncertainties) for the parameters
a_std_dev, b_std_dev = np.sqrt(np.diag(covariance))

In [14]:
print(f'BAYES a_fit (gradient in NF / Day): {a_fit:.2e} +- {a_std_dev:.2e} ({a_fit/a_std_dev:.2e} sigma)')
print(f'BAYES b_fit (intercept in NF): {b_fit:.2e} +- {b_std_dev:.2e}')

print(f'BAYES a_fit (gradient in NF / Year): {a_fit*365.25:.2e} +- {a_std_dev*365.35:.2e} ({a_fit/a_std_dev:.2e} sigma)')


BAYES a_fit (gradient in NF / Day): 4.39e-05 +- 2.30e-05 (1.91e+00 sigma)
BAYES b_fit (intercept in NF): 2.45e-01 +- 2.21e-02
BAYES a_fit (gradient in NF / Year): 1.60e-02 +- 8.41e-03 (1.91e+00 sigma)


### CALCULATE THE CONFIDENCE INTERVAL FOR THE BAYESIAN BEST FIT LINE

In [15]:
n_term = len(unzipped[1])
#term_2 = (x* - x_)**2

y_sum = 0.0

for n in range(n_term):
    y_sum += (unzipped[1][n] - (a_fit*mjd_delta[n] + b_fit))**2.0


s_y_x = ((y_sum)/(n_term-2.0))**0.5

In [16]:
def s_y_hat_func(x_wanted,x_data):
    
    term_1 = 1/len(x_data)
    
    term_2 = (x_wanted - np.mean(x_data))**2
    
    term_3 = 0.0
    for n in range(len(x_data)):
        term_3 += (x_data[n] - np.mean(x_data))**2.0
    
    full_term = (term_1 + (term_2/term_3))**0.5
    
    return full_term

In [17]:
mjd_delta = [mjd-unzipped[0][0] for mjd in unzipped[0]]
fit_unc = np.linspace(np.min(mjd_delta),np.max(mjd_delta), 100)

s_y_hat = []

for f in fit_unc:
    s_y_hat.append(s_y_hat_func(f,mjd_delta))
    
conf_interv = np.array(s_y_hat)*s_y_x

### TAKE UNCERTAINTY FROM HISTOGRAM SCALING TECHNIQUE AND COMBINE WITH BINOMAL UNCERTAINTY:

In [ ]:
# UNCERTAINTIES FOR THE HISTOGRAM SCALING METHOD

hs_and_binom_unc = []
hs_unc = []

for d in range(len(unzipped[2])):
    hs_unc.append(np.sqrt(unzipped[6][d]*n_prof_list[d]) / n_prof_list[d])
    hs_and_binom_unc.append(np.sqrt(binom_unc_nf_list[d]**2 + hs_unc[d]**2))

hs_and_binom_unc = np.array(hs_and_binom_unc)
hs_unc = np.array(hs_unc)

print(hs_and_binom_unc)

### CALCULATE THE BEST FIT LINE FOR THE HISTOGRAM SCALING TECHNIQUE:

In [ ]:
# Perform the curve fit
params_w, covariance_w = curve_fit(linear_model, mjd_delta, unzipped[6], sigma=hs_and_binom_unc, absolute_sigma=True)

# Extract the best-fit parameters
a_fit_w, b_fit_w = params_w

# Calculate the standard deviations (uncertainties) for the parameters
a_std_dev_w, b_std_dev_w = np.sqrt(np.diag(covariance_w))

In [ ]:
print(f'HS a_fit (gradient in NF / Day): {a_fit_w:.2e} +- {a_std_dev_w:.2e}')
print(f'HS b_fit (intercept): {b_fit_w:.2e} +- {b_std_dev_w:.2e}')

print(f'HS a_fit (gradient in NF / Year): {a_fit_w*365.25:.2e} +- {a_std_dev_w*365.35:.2e} ({a_fit_w/a_std_dev_w:.2e} sigma)')

### CALCULATE THE CONFIDENCE INTERVAL FOR THE HISTOGRAM SCALING BEST FIT LINE

In [21]:
n_term_w = len(unzipped[6])

y_sum_w = 0.0

for n_w in range(n_term_w):
    y_sum_w += (unzipped[6][n_w] - (a_fit_w * mjd_delta[n_w] + b_fit_w))**2.0

s_y_x_w = ((y_sum_w) / (n_term_w - 2.0))**0.5

In [22]:
conf_interv_w = np.array(s_y_hat)*s_y_x_w

### FIND UPPER AND LOWER LIMITS FOR THE NULLING FRACTION PLOTS BELOW:

In [ ]:
### ylims for both NF plots:
highest = max(np.max(unzipped[1]+uncertainties[1]), np.max(unzipped[6]+hs_and_binom_unc)) 
lowest = min(np.min(unzipped[1]-uncertainties[0]), np.min(unzipped[6]-hs_and_binom_unc)) 
buffer = 0.02 * (highest-lowest)

ylim_upp = highest + buffer
ylim_low = lowest - buffer

print(highest,lowest)
#print(unzipped[1]-uncertainties[0],unzipped[6]-hs_and_binom_unc)

### COVARIANCE-BASED 2σ CONFIDENCE INTERVALS

In [24]:
# Covariance-based 2σ confidence band for the Bayesian fit
x = np.linspace(np.min(mjd_delta), np.max(mjd_delta), 200)
y_hat = a_fit * x + b_fit
J = np.vstack([x, np.ones_like(x)])
var_y = np.einsum('ij,jk,ik->i', J.T, covariance, J.T)
sigma_y = np.sqrt(var_y)
lower_2sigma = y_hat - 2 * sigma_y
upper_2sigma = y_hat + 2 * sigma_y
x_abs = x + unzipped[0][0]

In [25]:
# Compute best-fit line and 2σ band for the weighted fit (bottom panel)

# Create a grid of x values (same spacing as before)
x_w = np.linspace(np.min(mjd_delta), np.max(mjd_delta), 200)

# Predicted line (best fit)
y_hat_w = a_fit_w * x_w + b_fit_w

# 1σ uncertainty for the mean response
J_w = np.vstack([x_w, np.ones_like(x_w)])             # shape (2, N)
var_y_w = np.einsum('ij,jk,ik->i', J_w.T, covariance_w, J_w.T)  # diag(J^T Cov J)
sigma_y_w = np.sqrt(var_y_w)

# 2σ confidence band
lower_2sigma_w = y_hat_w - 2 * sigma_y_w
upper_2sigma_w = y_hat_w + 2 * sigma_y_w

# Convert to absolute MJD for plotting (match the bottom panel’s x-axis)
x_abs_w = x_w + unzipped[0][0]


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import datetime
import matplotlib.gridspec as gridspec

def mjd_to_date(mjd):
    jd = mjd + 2400000.5
    return datetime.datetime(1858, 11, 17) + datetime.timedelta(days=jd - 2400000.5)

def first_day_of_year_mjd(year):
    first_day = datetime.datetime(year, 1, 1)
    jd = (first_day - datetime.datetime(1858, 11, 17)).days + 2400000.5
    return jd - 2400000.5

fig = plt.figure(figsize=(20, 15))
gs = gridspec.GridSpec(4, 1, height_ratios=[2, 2, 7, 7], hspace=0.0)

# Top panel — number of rotations
ax0 = plt.subplot(gs[0])
plt.plot(unzipped[0], unzipped[5], 'kx', markersize=10, mew=2)
plt.grid(True)
ax0.set_ylim(np.nanmin(unzipped[5]) - ((np.nanmax(unzipped[5]) - np.nanmin(unzipped[5])) * 0.2),
             np.nanmax(unzipped[5]) + ((np.nanmax(unzipped[5]) - np.nanmin(unzipped[5])) * 0.2))
plt.gca().set_xticklabels([])

ax0_top = ax0.twiny()
ax0_top.set_xlim(ax0.get_xlim())
years = [mjd_to_date(mjd).year for mjd in unzipped[0]]
unique_years = sorted(set(years))
min_year, max_year = min(unique_years), max(unique_years)
unique_years = list(range(min_year + 1, max_year + 1))
first_day_mjds = [first_day_of_year_mjd(year) for year in unique_years]
ax0_top.set_xticks(first_day_mjds)
ax0_top.set_xticklabels([str(year) for year in unique_years])
plt.setp(ax0_top.get_xticklabels(), rotation=0, ha='center')
ax0_top.tick_params(axis='x', labelsize=20)

# Second panel — S/N proxy
ax1 = plt.subplot(gs[1])
plt.plot(unzipped[0], unzipped[4], 'kx', markersize=10, mew=2)
plt.grid(True)
ax1.set_ylim(np.nanmin(unzipped[4]) - ((np.nanmax(unzipped[4]) - np.nanmin(unzipped[4])) * 0.2),
             np.nanmax(unzipped[4]) + ((np.nanmax(unzipped[4]) - np.nanmin(unzipped[4])) * 0.2))
plt.gca().set_xticklabels([])

# Third panel — BPE nulling fraction
ax3 = plt.subplot(gs[2])
plt.gca().set_xticklabels([])
plt.rcParams['xtick.labelsize'] = 16
plt.rcParams['ytick.labelsize'] = 16
plt.errorbar(unzipped[0], unzipped[1], yerr=uncertainties, fmt='kx', markersize=20, capsize=10, lw=3, mew=3)
plt.errorbar(unzipped[0], unzipped[1], yerr=bayes_uncertainties, fmt='k,', capsize=5, lw=3, mew=3)
plt.ylim(ylim_low, ylim_upp)
plt.grid(True)

x_range = np.linspace(min(unzipped[0]), max(unzipped[0]), 100)
x_range_new = np.linspace(min(mjd_delta), max(mjd_delta), 100)
plt.plot(x_range, linear_model(x_range_new, a_fit, b_fit), color='k', linewidth=5, linestyle='--', alpha=0.5, zorder=-100)
plt.fill_between(x_abs, lower_2sigma, upper_2sigma, color='red', alpha=0.2, label='2σ band (covariance)')

# Fourth panel — Histogram Scaling nulling fraction
ax4 = plt.subplot(gs[3])
plt.rcParams['xtick.labelsize'] = 16
plt.rcParams['ytick.labelsize'] = 16
plt.errorbar(unzipped[0], unzipped[6], yerr=hs_and_binom_unc, fmt='kx', markersize=20, capsize=10, lw=3, mew=3, zorder=1)
plt.errorbar(unzipped[0], unzipped[6], yerr=hs_unc, fmt='k,', capsize=5, lw=3, mew=3, zorder=1)
plt.ylim(ylim_low, ylim_upp)
plt.grid(True)

plt.plot(x_range, linear_model(x_range_new, a_fit_w, b_fit_w), color='k', linewidth=5, linestyle='--', alpha=0.5, zorder=3)
plt.fill_between(x_abs_w, lower_2sigma_w, upper_2sigma_w, color='red', alpha=0.2, label='2σ band (covariance, weighted)', zorder=2)

for ax in [ax0, ax1, ax3, ax4]:
    ax.tick_params(axis='both', labelsize=20)
    for spine in ax.spines.values():
        spine.set_linewidth(2)

fig.text(0.053, 0.845, 'No. of\nRotations', ha='center', va='center', rotation='vertical', size=22)
fig.text(0.053, 0.757, 'S/N\nProxy', ha='center', va='center', rotation='vertical', size=22)
fig.text(0.065, 0.42, 'Nulling Fraction', ha='center', va='center', rotation='vertical', size=22)
fig.text(0.504, 0.934, 'Year', ha='center', va='center', rotation='horizontal', size=23)
fig.text(0.504, 0.07, 'Modified Julian Date', ha='center', va='center', rotation='horizontal', size=22)

plt.savefig(f'{pulsar_name}_main_nf_evolution_start.png', facecolor='white')
plt.show()